# Notebook 08: Vision Transformer (ViT)

## Why This Matters

The Vision Transformer showed that the transformer architecture — designed for text — could be applied to images with minimal modification and eventually surpass CNNs at scale. ViT is now the backbone of multimodal models (CLIP, GPT-4V, Gemini), image generation models (DiT — Diffusion Transformer), and the best image classification models. Understanding ViT means understanding how the "patches as tokens" abstraction works, what differences from text transformers matter, and why ViT needs more data than CNNs but scales better. This is also the foundation for understanding how models like DALL-E 3 and Stable Diffusion 3 work at the architecture level.

## Background

**From CNNs to ViT:** Convolutional Neural Networks exploit two structural priors: local connectivity (nearby pixels are correlated) and translation equivariance (a feature detector works the same regardless of position). These are excellent inductive biases for small/medium data but limit flexibility at large scale.

ViT (Dosovitskiy et al. 2020) ditched these inductive biases: instead, it divides an image into non-overlapping patches (e.g., 16×16 pixels), flattens each patch into a vector, and processes them as a sequence of tokens with standard transformer self-attention.

**The trade-off:** Without CNN's local inductive bias, ViT needs more data to learn spatial structure from scratch. On small datasets (ImageNet-1K alone), ViT performs worse than CNNs of equivalent size. At scale (ImageNet-21K or JFT-300M pre-training), ViT significantly outperforms CNNs of equivalent FLOPs.

**Key insight:** With enough data, the model learns its own local structure — some attention heads learn to attend to adjacent patches, others learn global patterns. The architecture doesn't need to hard-code these priors.

In [ ]:
%pip install -q torch torchvision numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. Patch Embedding: Splitting Images Into Tokens

An image of shape $(C, H, W)$ is split into $N = (H/P) \times (W/P)$ non-overlapping patches of size $P \times P \times C$. Each patch is then flattened and projected to dimension $d_{\text{model}}$:

$$N = \frac{H \cdot W}{P^2}, \quad x_{\text{patch}_i} \in \mathbb{R}^{P^2 C}$$

**Efficient implementation via Conv2d:** A convolution with kernel size $P$ and stride $P$ is mathematically equivalent to patch extraction + linear projection, and is highly optimized on GPU:
- Kernel size: $(P, P)$  
- Stride: $(P, P)$  
- Output channels: $d_{\text{model}}$

This single Conv2d operation handles both patch extraction and projection in one shot.

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Split image into patches and embed them.
    Input: (B, C, H, W)
    Output: (B, N, d_model) where N = (H/P)*(W/P)
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3, d_model=192):
        super().__init__()
        assert img_size % patch_size == 0, "Image size must be divisible by patch size"
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        # Conv2d with kernel=patch_size, stride=patch_size = efficient patch extraction + projection
        self.projection = nn.Conv2d(in_channels, d_model,
                                    kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.projection(x)           # (B, d_model, H/P, W/P)
        x = x.flatten(2)                 # (B, d_model, N)
        x = x.transpose(1, 2)           # (B, N, d_model)
        return x

# Test with CIFAR-10 dimensions (32x32, 3 channels)
img_size, patch_size, d_model = 32, 4, 192
patch_embed = PatchEmbedding(img_size, patch_size, in_channels=3, d_model=d_model)
images = torch.randn(8, 3, img_size, img_size)
patches = patch_embed(images)
print(f"Input image shape: {images.shape}")
print(f"Patch embedding shape: {patches.shape}")
print(f"Number of patches: {patch_embed.n_patches} (= {img_size//patch_size}×{img_size//patch_size})")
print(f"Each patch represents {patch_size}×{patch_size}×3 = {patch_size**2*3} raw values → {d_model} dims")

## 2. CLS Token and 2D Positional Embedding

**CLS token:** A learnable embedding prepended to the patch sequence. After transformer processing, the CLS token's output serves as the image representation for classification (same as BERT). Alternatively, global average pooling over all patch tokens works equally well or better.

**Positional embedding challenge:** Unlike text (1D sequence), images have 2D spatial structure. ViT uses 1D learned positional embeddings (one embedding per patch position, treating the 2D grid as a 1D sequence). Surprisingly, this works well — the model can learn the 2D structure from data. More sophisticated 2D sinusoidal PEs don't consistently improve results.

**Interpolating positional embeddings:** When fine-tuning ViT at a different resolution than pre-training, the number of patches changes. The solution: bilinearly interpolate the 2D positional embeddings from the pre-training grid to the new grid.

In [ ]:
class ViT(nn.Module):
    """Vision Transformer for image classification (CIFAR-10 scale)."""
    def __init__(self, img_size=32, patch_size=4, in_channels=3, n_classes=10,
                 d_model=192, n_heads=3, n_layers=9, d_ff=None, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        n_patches = self.patch_embed.n_patches

        # CLS token: a learnable vector prepended to patch sequence
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        # Learned positional embeddings: one per patch + one for CLS
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, d_model) * 0.02)
        self.dropout = nn.Dropout(dropout)

        # Transformer encoder layers
        d_ff = d_ff or 4 * d_model
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True  # Pre-LN
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

        # Classification head
        self.head = nn.Linear(d_model, n_classes)

        self._init_weights()

    def _init_weights(self):
        # ViT uses truncated normal initialization
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        B = x.size(0)
        # Extract patch embeddings
        x = self.patch_embed(x)               # (B, N, d_model)
        # Prepend CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, N+1, d_model)
        x = self.dropout(x + self.pos_embed)

        # Transformer (no mask needed — bidirectional)
        x = self.transformer(x)
        x = self.norm(x)

        # Classification from CLS token (position 0)
        cls_output = x[:, 0, :]
        return self.head(cls_output)

vit = ViT().to(device)
images = torch.randn(4, 3, 32, 32).to(device)
logits = vit(images)
print(f"ViT output shape: {logits.shape}")  # (4, 10)
total_params = sum(p.numel() for p in vit.parameters())
print(f"ViT parameters: {total_params:,}")  # ~3.7M for tiny ViT

## 3. Training ViT on CIFAR-10

In [ ]:
# Load CIFAR-10 with standard augmentation
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10('./data', train=True, download=True,
                                         transform=transform_train)
testset  = torchvision.datasets.CIFAR10('./data', train=False, download=True,
                                         transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=256, shuffle=True,
                                           num_workers=2, pin_memory=True)
testloader  = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False,
                                           num_workers=2, pin_memory=True)
print(f"Train batches: {len(trainloader)}, Test batches: {len(testloader)}")
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

In [ ]:
# Train ViT for a few epochs to show it learning
model = ViT(img_size=32, patch_size=4, d_model=192, n_heads=3, n_layers=6).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)
    return correct / total

n_epochs = 10
history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
print(f"Training ViT for {n_epochs} epochs...")
for epoch in range(n_epochs):
    tr_loss, tr_acc = train_epoch(model, trainloader, optimizer, criterion, device)
    te_acc = eval_epoch(model, testloader, device)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['test_acc'].append(te_acc)
    print(f"Epoch {epoch+1:2d}: loss={tr_loss:.4f}, train_acc={tr_acc:.3f}, test_acc={te_acc:.3f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, n_epochs+1)
axes[0].plot(epochs, history['train_loss'], 'b-o', markersize=5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Training Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_acc'], 'b-o', markersize=5, label='Train')
axes[1].plot(epochs, history['test_acc'],  'r-o', markersize=5, label='Test')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.suptitle("ViT Training on CIFAR-10 (tiny model)", fontsize=12)
plt.tight_layout()
plt.savefig("vit_training.png", dpi=120, bbox_inches='tight')
plt.show()

## 4. Key Differences from Text Transformers

| Aspect | Text Transformer | Vision Transformer |
|---|---|---|
| Input | Token IDs (integers) | Image patches (continuous floats) |
| Position | Absolute or RoPE | Learned 1D (treating 2D grid as sequence) |
| Masking | Causal mask (decoder) | None (bidirectional, no future to mask) |
| CLS usage | Optional | Standard for classification |
| Scale dependence | Works at any scale | Needs large data to outperform CNNs |
| Patch size | N/A | Controls speed/accuracy tradeoff |

**DeiT (Data-efficient ViT, Touvron et al. 2021):** Achieved competitive ImageNet-1K results from scratch using knowledge distillation from a CNN teacher. Added a "distillation token" (analogous to CLS) that learns to mimic the CNN teacher's predictions. This closed the data efficiency gap with CNNs, enabling ViT training without massive pre-training datasets.

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Patches as tokens | Divide image into $P \times P$ patches; each patch = one token; no CNN needed |
| $N = HW/P^2$ patches | Patch size $P$ trades off sequence length vs spatial resolution |
| Conv2d patching | kernel=stride=$P$; efficient equivalent of flatten+project |
| CLS token | Prepended; after transformer, CLS output = image representation for classification |
| 1D positional PE | Works despite 2D structure; model learns spatial relationships from data |
| Interpolation for resize | Bilinear interpolate PE grid when changing input resolution |
| ViT vs CNN | ViT needs more data; scales better; no local inductive bias |
| DeiT | Distillation from CNN teacher enables ImageNet-1K ViT without massive pre-training |
| DiT | ViT applied to latent diffusion; backbone of DALL-E 3 / Stable Diffusion 3 |